# Session 9: Docker, CI/CD & Production Deployment

Deploying Ask IRA to production using Docker, Kubernetes, Railway, and GitHub Actions CI/CD.

## 1. Multi-Stage Docker Build

In [ ]:
print("=" * 60)
print("Ask IRA Deployment Architecture")
print("=" * 60)
print("""
Build Stage (python:3.11-slim):
  - Install build-essential, compilers
  - pip install ask-ira[all]  → /install prefix

Runtime Stage (python:3.11-slim):
  - Copy /install from builder
  - Create non-root 'askira' user
  - HEALTHCHECK on /health
  - CMD: uvicorn src.main:app --workers 2
"")

### Dockerfile Highlights

- **Multi-stage**: Builder (with compilers) → Runtime (slim, no build tools)
- **Non-root user**: Runs as `askira` user, not root
- **HEALTHCHECK**: Every 30s, curl `/health`, 3 retries, 15s startup
- **Port**: 8000, 2 workers

## 2. Docker Compose — Full Stack

In [ ]:
compose_services = {
    "api": {
        "image": "ask-ira:latest",
        "ports": "8000:8000",
        "depends_on": ["postgres", "redis"],
        "healthcheck": "/health",
        "resources": {"memory": "2G limit, 512M reserved"},
    },
    "postgres": {
        "image": "postgres:16-alpine",
        "purpose": "Enterprise DB + LangGraph checkpointing",
        "healthcheck": "pg_isready",
    },
    "redis": {
        "image": "redis:7-alpine",
        "purpose": "Response caching + rate limiting",
        "healthcheck": "redis-cli ping",
    },
    "seed": {
        "profile": "seed (manual run)",
        "command": "python -m scripts.seed_data",
    },
}

for name, config in compose_services.items():
    print(f"  [{name}]")
    for k, v in config.items():
        print(f"    {k}: {v}")
    print()

print("Commands:")
print("  docker compose up -d              # Start all services")
print("  docker compose --profile seed run seed  # Seed data")

### Docker Commands

```bash
# Development
docker compose -f deployment/docker-compose.yml up -d

# With monitoring (Prometheus + Grafana)
docker compose -f deployment/docker-compose.yml \
  -f deployment/docker-compose.monitoring.yml up -d

# Seed vector store
docker compose --profile seed run seed

# View logs
docker compose logs -f api

# Scale API
docker compose up -d --scale api=3
```

## 3. Kubernetes Deployment

In [ ]:
k8s_manifests = {
    "deployment.yaml": "3 replicas, rolling update, resource limits",
    "hpa.yaml": "CPU > 70% or memory > 80% → scale 2-10 pods",
    "ingress.yaml": "nginx ingress with TLS termination",
    "configmap.yaml": "Non-sensitive env vars",
    "secrets.yaml": "API keys (Base64 encoded)",
}

print("Kubernetes manifests:")
for name, desc in k8s_manifests.items():
    print(f"  deployment/kubernetes/{name}")
    print(f"    → {desc}")
    print()

### K8s Deploy

```bash
kubectl apply -f deployment/kubernetes/
kubectl get pods -l app=ask-ira
kubectl get hpa ask-ira-hpa
kubectl get ingress ask-ira-ingress
```

## 4. Railway Deployment

In [ ]:
railway_config = {
    "builder": "DOCKERFILE",
    "dockerfilePath": "deployment/Dockerfile",
    "numReplicas": 1,
    "restartPolicy": "ON_FAILURE",
    "healthcheckPath": "/health",
    "healthcheckTimeout": 30,
}

print("Railway deployment config:")
for k, v in railway_config.items():
    print(f"  {k}: {v}")

print("\nDeploy command:")
print("  railway up --service ask-ira")

## 5. GitHub Actions CI/CD

In [ ]:
ci_pipeline = {
    "lint": "Ruff lint → py311",
    "test": "pytest matrix (3.11, 3.12) + coverage",
    "docker_build": "Docker Buildx multi-arch smoke test",
}

cd_pipeline = {
    "railway": "Trigger: push to main → railway up",
    "docker_hub": "Trigger: push to main → build + push multi-arch",
}

print("CI Pipeline:")
for stage, desc in ci_pipeline.items():
    print(f"  {stage}: {desc}")

print("\nCD Pipeline:")
for target, trigger in cd_pipeline.items():
    print(f"  {target}: {trigger}")

print("\nSecrets required:")
print("  RAILWAY_TOKEN, DOCKER_USERNAME, DOCKER_PASSWORD")

### Workflow Files

- `.github/workflows/ci.yml` — lint → test matrix (py311, py312) → Docker smoke test
- `.github/workflows/deploy.yml` — Railway + Docker Hub multi-arch push on main

In [ ]:
print("Session 9 complete — deployment targets: Docker, K8s, Railway, Docker Hub")